# memory
> Semantic Memory Module for Linked Data Agents

In [ ]:
#| default_exp memory

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from pyld import jsonld
import httpx
from rdflib import Graph, Dataset, URIRef, Literal, Namespace
from rdflib.namespace import RDF, RDFS
from datetime import datetime, timedelta
from fastcore.basics import patch
from fastcore.test import  *
from claudette import Chat, toolloop, models
import uuid
import hashlib
import re
import copy

# Semantic Memory Class Initialization
Define the `__init__` method for the `SemanticMemory` class, setting up core data structures like the dataset, indices, context registry, and HTTP client.

In [ ]:
#| export
class SemanticMemory:
    def __init__(self):
        """Initialize the semantic memory system with all required attributes"""
        # Core data structures from original implementation
        self.dataset = Dataset()
        self.default_graph = self.dataset.graph(URIRef("urn:x-arq:DefaultGraph"))
        self.contexts = {} # Legacy context cache, prefer context_registry
        self.indices = {
            "by_id": {},
            "by_type": {},
            "by_index": {}, # Placeholder for future index types
            "by_label": {},
            "by_description": {}
        }
        self.uri_cache = {} # Cache for dereferenced URIs
        self.client = httpx.Client(follow_redirects=True)
        
        # Context registry (Enhanced)
        self.context_registry = {
            # "context_id": {
            #     "context": {...},  # The actual context object
            #     "source": "...",   # Where this context came from (URI or generated ID)
            #     "parent": "...",   # Parent context ID if this is a scoped context
            #     "scoped_contexts": {},  # Map of term -> context_id for terms with scoped contexts
            #     "used_by": set()   # Set of entity IDs that use this context
            # }
        }
        
        # Entity context tracking (Enhanced)
        self.entity_contexts = {
            # "entity_id": {
            #     "primary_context": "context_id",
            #     "property_contexts": {
            #         "property1": "context_id",
            #         "property2": "context_id"
            #     }
            # }
        }
        
        # Storage for original (non-expanded) JSON-LD (Enhanced)
        self.original_data = {
            # "entity_id": {...}  # Original JSON-LD data
        }
        
        # Graph tracking (for named graphs and @graph structures)
        self.graph_entities = {}  # Maps graph URNs/IDs to lists of entity URNs/IDs
        self.entity_graphs = {}   # Maps entity URNs/IDs to the graphs they belong to

        # URI to URN mapping (if needed, though URNs are less common now)
        self.uri_to_urn = {}
        self.urn_to_uri = {} # Inverse mapping

        # Alias common indices for easier access
        self.by_id = self.indices["by_id"]
        self.by_type = self.indices["by_type"]

# Context Handling Functions
Implement functions for managing JSON-LD contexts: `_hash_json`, `fetch_context`, `_register_contexts`, `_get_context_id`, `manage_context`. These handle fetching, caching, registering, and identifying contexts, including scoped contexts.

In [ ]:
#| export
@patch
def _hash_json(self:SemanticMemory, obj):
    "Create a hash of a JSON object for context registry lookups"
    import hashlib, json
    if isinstance(obj, dict) or isinstance(obj, list):
        json_str = json.dumps(obj, sort_keys=True)
        return hashlib.md5(json_str.encode('utf-8')).hexdigest()
    return str(obj)

In [ ]:
#| export
@patch
def fetch_context(self:SemanticMemory, context_uri):
    """Fetch and cache a JSON-LD context (Legacy, prefer manage_context)"""
    if context_uri in self.contexts:
        return self.contexts[context_uri]["document"]
        
    # For simplicity in our initial implementation, we'll handle local contexts
    # and leave remote fetching for later
    if isinstance(context_uri, dict):
        # It's already a context object
        context_id = self._get_context_id(context_uri) # Use stable ID
        self.contexts[context_id] = { # Use stable ID as key
            "document": context_uri,
            "last_fetched": datetime.now().isoformat(),
            "dependencies": []
        }
        return context_uri
    
    # For now, just return a simple context if URI not found
    # We'll implement proper HTTP fetching in a later step (see manage_context)
    print(f"Warning: Fetching context {context_uri} not fully implemented in fetch_context. Using empty context.")
    simple_context = {}
    self.contexts[context_uri] = {
        "document": simple_context,
        "last_fetched": datetime.now().isoformat(),
        "dependencies": []
    }
    return simple_context

In [ ]:
#| export
@patch
def _register_contexts(self:SemanticMemory, data, parent_context_id=None):
    """Register all contexts in a JSON-LD document, including scoped contexts (Enhanced)"""
    if not isinstance(data, dict):
        return None
    
    # Handle the main context
    if "@context" in data:
        context = data["@context"]
        context_id = self._get_context_id(context)
        
        # If this context is already registered, just return its ID
        if context_id in self.context_registry:
            # Update parent relationship if this is a scoped context
            if parent_context_id:
                self.context_registry[context_id]["parent"] = parent_context_id
            return context_id
        
        # Register the new context
        self.context_registry[context_id] = {
            "context": context,
            "source": "document",  # Or could be URI if it's a remote context
            "parent": parent_context_id,
            "scoped_contexts": {},
            "used_by": set()
        }
        
        # Look for scoped contexts within this context
        if isinstance(context, dict):
            for term, value in context.items():
                # Check if the value defines a scoped context
                if isinstance(value, dict) and "@context" in value:
                    # This term has a scoped context
                    # The scoped context itself is defined within the 'value' dictionary
                    scoped_context_definition = {"@context": value["@context"]}
                    scoped_context_id = self._register_contexts(scoped_context_definition, context_id)
                    if scoped_context_id: # Ensure registration was successful
                         self.context_registry[context_id]["scoped_contexts"][term] = scoped_context_id
        
        return context_id
    
    return None

In [ ]:
#| export
@patch
def _get_context_id(self:SemanticMemory, context):
    """Generate a stable ID for a context"""
    # If context is a URI string, use it directly
    if isinstance(context, str):
        return context
    
    # If it's a list, concatenate the IDs of each component
    if isinstance(context, list):
        # Recursively get IDs and filter out None values
        component_ids = [self._get_context_id(c) for c in context]
        valid_ids = [cid for cid in component_ids if cid]
        return "context:list:" + "_".join(valid_ids)
    
    # For a context object, use a hash of its JSON representation
    if isinstance(context, dict):
        try:
            context_str = json.dumps(context, sort_keys=True)
            import hashlib
            return "context:dict:" + hashlib.md5(context_str.encode()).hexdigest()
        except TypeError: # Handle non-serializable content if necessary
             return f"context:dict:error:{id(context)}"

    # Fallback for other types (though less common for contexts)
    return f"context:unknown:{id(context)}"

In [ ]:
#| export
@patch
def manage_context(self:SemanticMemory, context_uri, force_refresh=False, ttl_seconds=86400):
    """
    Fetch, cache, and process a JSON-LD context (More robust than fetch_context)
    
    Args:
        context_uri: URI or dictionary object of the context to manage
        force_refresh: Whether to force a refresh even if cached
        ttl_seconds: Time-to-live in seconds for context cache entries (default: 1 day)
    
    Returns:
        The processed context dictionary, or None if fetching/processing fails
    """
    now = datetime.now()
    
    # Handle context objects directly
    if isinstance(context_uri, dict):
        context_id = self._get_context_id(context_uri)
        if context_id not in self.context_registry or force_refresh:
             self.context_registry[context_id] = {
                 "context": context_uri,
                 "source": f"dict:{context_id}",
                 "fetched_at": now,
                 "expires_at": now + timedelta(seconds=ttl_seconds),
                 "dependencies": [],
                 "parent": None, # Dictionaries provided directly don't have parents unless registered via _register_contexts
                 "scoped_contexts": {},
                 "used_by": set()
             }
             # Recursively register any scoped contexts within this dictionary
             self._register_contexts({"@context": context_uri})
        return self.context_registry[context_id]["context"]

    # Context is a URI (string)
    context_id = context_uri # Use URI as the ID

    # Check if we already have this context cached and valid
    if not force_refresh and context_id in self.context_registry:
        context_entry = self.context_registry[context_id]
        if context_entry.get("expires_at") and context_entry["expires_at"] > now:
            return context_entry["context"]
            
    # Need to fetch the context from the URI
    try:
        # Use our existing dereference method, which handles caching and content negotiation
        # Note: dereference_uri adds the *data* to memory, not just the context
        # We might want a dedicated context fetcher later, but this works for now.
        context_data = self.dereference_uri(context_uri, force_refresh, ttl_seconds)
        
        if not context_data or "error" in context_data:
             print(f"Error dereferencing context URI {context_uri}: {context_data.get('error', 'Unknown error')}")
             return None

        # Extract the actual context from the fetched data
        # It might be the whole document or within a @context key
        if isinstance(context_data, dict) and "@context" in context_data:
            context = context_data["@context"]
        elif isinstance(context_data, dict): # Assume the whole doc is the context if no @context key
             context = context_data
        else:
             print(f"Warning: Fetched context data for {context_uri} is not a dictionary.")
             context = {} # Fallback to empty context

        # Process context dependencies (@import) - Simplified
        dependencies = []
        if isinstance(context, dict) and "@import" in context:
            import_uri = context["@import"]
            print(f"Note: @import found for {import_uri}, processing recursively.")
            imported_context = self.manage_context(import_uri, force_refresh, ttl_seconds)
            dependencies.append(import_uri)
            
            # Merge imported context with current one (basic merge)
            if isinstance(imported_context, dict):
                merged_context = imported_context.copy()
                merged_context.update(context) # Current context overrides imported
                context = merged_context
            del context["@import"] # Remove @import after processing

        # Store in our context registry
        self.context_registry[context_id] = {
            "context": context,
            "source": context_uri,
            "fetched_at": now,
            "expires_at": now + timedelta(seconds=ttl_seconds),
            "dependencies": dependencies,
            "parent": None, # Fetched contexts are top-level unless registered via _register_contexts
            "scoped_contexts": {},
            "used_by": set()
        }
        # Recursively register any scoped contexts defined within this fetched context
        self._register_contexts({"@context": context})

        return context
    
    except Exception as e:
        print(f"Exception managing context {context_uri}: {e}")
        # Cache the error too, but with shorter TTL
        self.context_registry[context_id] = {
            "context": {"error": str(e)},
            "source": context_uri,
            "fetched_at": now,
            "expires_at": now + timedelta(seconds=min(ttl_seconds, 300)),
            "dependencies": [],
            "parent": None,
            "scoped_contexts": {},
            "used_by": set()
        }
        return None # Return None on error

# Data Addition Functions
Implement functions for adding data: `add_jsonld`, `add_jsonld_with_graph`, `create_entity_with_uuid`. These handle adding individual entities or graphs, preserving context, generating IDs, and updating the RDF dataset.

In [ ]:
#| export
@patch
def add_jsonld(self:SemanticMemory, data, context=None):
    """Add JSON-LD data while preserving context information (Enhanced)"""
    
    # Make a copy to avoid modifying the original input data
    data_copy = copy.deepcopy(data)

    # Apply external context if provided
    if context:
        # Fetch/manage the external context first
        managed_context = self.manage_context(context)
        if managed_context:
            if isinstance(data_copy, dict):
                if "@context" in data_copy:
                    # Merge contexts: external context first, then internal
                    existing_context = data_copy["@context"]
                    if isinstance(existing_context, list):
                        data_copy["@context"] = [managed_context] + existing_context
                    else:
                        data_copy["@context"] = [managed_context, existing_context]
                else:
                    data_copy["@context"] = managed_context
            # If data_copy is a list, context applies to the container (less common)
            # We might need more sophisticated handling for lists with context.

    # Process and register contexts defined within the data itself
    primary_context_id = None
    if isinstance(data_copy, dict) and "@context" in data_copy:
        primary_context_id = self._register_contexts(data_copy)

    # Determine the entity ID
    entity_id = None
    if isinstance(data_copy, dict) and "@id" in data_copy:
        entity_id = data_copy["@id"]
    elif isinstance(data_copy, list) and len(data_copy) == 1 and isinstance(data_copy[0], dict) and "@id" in data_copy[0]:
         # Handle case where input is a list containing a single entity dict
         entity_id = data_copy[0]["@id"]
    else:
        # Generate a UUID-based URN if no ID exists or if it's not a dict with @id
        entity_id = f"urn:uuid:{uuid.uuid4()}"
        if isinstance(data_copy, dict):
            data_copy["@id"] = entity_id
        elif isinstance(data_copy, list) and len(data_copy) == 1 and isinstance(data_copy[0], dict):
             data_copy[0]["@id"] = entity_id
        # If it's a list of multiple items without IDs, handling is complex.
        # For now, we focus on single entities or graphs.

    # Store the original data (with potentially added context/ID)
    self.original_data[entity_id] = data_copy
    
    # Expand the data for RDF conversion and basic indexing
    # Use jsonld.expand with appropriate options if needed (e.g., base URI)
    try:
        # We need a document loader that uses our context management
        def custom_document_loader(url, options={}):
             context_doc = self.manage_context(url)
             if context_doc:
                 return {
                     "contextUrl": None, # Set to None to indicate context is embedded
                     "documentUrl": url,
                     "document": {"@context": context_doc}
                 }
             # Fallback for non-context URLs or errors
             # This part needs careful implementation based on pyld's expected format
             # For now, let's try fetching normally if manage_context fails
             try:
                 response = self.client.get(url, headers={"Accept": "application/ld+json, application/json"})
                 response.raise_for_status()
                 return {
                     "contextUrl": None,
                     "documentUrl": url,
                     "document": response.json()
                 }
             except Exception as e:
                 print(f"Custom loader failed for {url}: {e}")
                 raise jsonld.JsonLdError(f"Failed to load document: {url}", "jsonld.LoadDocumentError")

        # Expand using the custom loader
        expanded = jsonld.expand(data_copy, {"documentLoader": custom_document_loader})
    except Exception as e:
        print(f"Error expanding JSON-LD for entity {entity_id}: {e}")
        print(f"Data causing error: {json.dumps(data_copy, indent=2)[:500]}...")
        return None # Return None or handle error appropriately

    # Update our indices based on the expanded data
    self._update_indices(expanded) # Basic ID/Type index
    self._update_indices_with_labels(expanded) # Label/Description index
    
    # Convert expanded data to RDF and add to the default graph
    # Note: This adds to the *default* graph. For named graphs, use add_named_graph.
    try:
        g = Graph()
        g.parse(data=json.dumps(expanded), format='json-ld')
        self.default_graph += g
    except Exception as e:
        print(f"Error parsing expanded JSON-LD to RDF for entity {entity_id}: {e}")
        # Decide if we should proceed without adding to RDF graph or stop

    # Track which context(s) this entity uses
    if entity_id not in self.entity_contexts:
        self.entity_contexts[entity_id] = {
            "primary_context": None,
            "property_contexts": {} # For future fine-grained tracking
        }
    
    # Record the primary context ID if one was registered
    if primary_context_id:
        self.entity_contexts[entity_id]["primary_context"] = primary_context_id
        if primary_context_id in self.context_registry:
             self.context_registry[primary_context_id]["used_by"].add(entity_id)
        else:
             print(f"Warning: Primary context ID {primary_context_id} not found in registry for entity {entity_id}")

    # Track graph membership if this entity is part of a graph structure (handled in add_jsonld_with_graph/add_named_graph)

    return expanded

In [ ]:
#| export
@patch
def add_jsonld_with_graph(self:SemanticMemory, data, context=None):
    """Enhanced version of add_jsonld that properly handles @graph structures while preserving contexts.

    Args:
        data: JSON-LD data to add (can be a single entity or a document with @graph)
        context: Optional context to apply externally

    Returns:
        A list of expanded entities if input was a @graph, or a single expanded entity otherwise.
        Returns None on error.
    """
    data_copy = copy.deepcopy(data)

    # Apply external context if provided
    if context:
        managed_context = self.manage_context(context)
        if managed_context:
            if isinstance(data_copy, dict):
                if "@context" in data_copy:
                    existing_context = data_copy["@context"]
                    if isinstance(existing_context, list):
                        data_copy["@context"] = [managed_context] + existing_context
                    else:
                        data_copy["@context"] = [managed_context, existing_context]
                else:
                    data_copy["@context"] = managed_context
            # Else: context applies to the container? Needs clarification.

    # Check if the input data uses the @graph keyword
    if isinstance(data_copy, dict) and '@graph' in data_copy:
        graph_nodes = data_copy['@graph']
        graph_context = data_copy.get('@context')
        graph_id = data_copy.get('@id') # Optional ID for the graph itself

        # Register the graph's context
        graph_context_id = None
        if graph_context:
            graph_context_id = self._register_contexts({"@context": graph_context})

        # Process the graph node itself if it has properties other than @graph, @context, @id
        graph_metadata = {k: v for k, v in data_copy.items() if k not in ['@graph', '@context', '@id']}
        if graph_id and graph_metadata:
             graph_metadata['@id'] = graph_id
             if graph_context: graph_metadata['@context'] = graph_context
             # Add the graph metadata as a separate entity
             self.add_jsonld(graph_metadata) # Recursive call for the graph node itself

        # Process each entity within the @graph array
        expanded_results = []
        for entity in graph_nodes:
            entity_copy = copy.deepcopy(entity)
            
            # Inherit graph context if entity doesn't have its own
            if graph_context and '@context' not in entity_copy:
                entity_copy['@context'] = graph_context
            
            # Add the entity using the regular add_jsonld method
            expanded_entity = self.add_jsonld(entity_copy) # Recursive call
            if expanded_entity:
                expanded_results.extend(expanded_entity) # add_jsonld returns a list

                # Track graph membership (if graph has an ID)
                entity_id = None
                if isinstance(expanded_entity, list) and len(expanded_entity) > 0:
                     entity_id = expanded_entity[0].get('@id') # Get ID from expanded form

                if graph_id and entity_id:
                    if graph_id not in self.graph_entities: self.graph_entities[graph_id] = []
                    if entity_id not in self.graph_entities[graph_id]: self.graph_entities[graph_id].append(entity_id)
                    
                    if entity_id not in self.entity_graphs: self.entity_graphs[entity_id] = []
                    if graph_id not in self.entity_graphs[entity_id]: self.entity_graphs[entity_id].append(graph_id)

        return expanded_results # Return list of expanded entities from the graph

    else:
        # If it's not a @graph structure, process it as a single entity
        expanded_entity = self.add_jsonld(data_copy) # Use the enhanced add_jsonld
        return expanded_entity # Returns list containing the single expanded entity or None

In [ ]:
#| export
@patch
def create_entity_with_uuid(self:SemanticMemory, data, type_uri=None, context=None):
    """Create a new entity with a UUID identifier and add it to memory"""
    
    # Generate a UUID-based URN
    entity_uuid = str(uuid.uuid4())
    entity_id = f"urn:uuid:{entity_uuid}"
    
    # Create the basic entity structure
    entity = {
        "@id": entity_id
    }
    
    # Add type if provided
    if type_uri:
        entity["@type"] = type_uri
    
    # Add any additional data properties
    if isinstance(data, dict):
        for key, value in data.items():
            # Avoid overwriting @id or adding @context directly here
            # Context should be passed separately or be part of the 'data' dict's own context
            if key not in ["@id", "@context"]:
                entity[key] = value
    
    # Add context if provided externally or if present in 'data'
    final_context = context
    if isinstance(data, dict) and "@context" in data:
         if final_context: # Merge if both exist
             # Simple merge: external context first
             existing_context = data["@context"]
             if isinstance(existing_context, list):
                 final_context = [final_context] + existing_context
             else:
                 final_context = [final_context, existing_context]
         else:
             final_context = data["@context"]

    # Add the entity to memory using add_jsonld, passing the context
    self.add_jsonld(entity, context=final_context)
    
    # Return the generated ID
    return entity_id

# Indexing Functions
Implement functions for updating internal indices: `_update_indices`, `_update_indices_with_labels`. These populate indices based on ID, type, labels, and descriptions from expanded JSON-LD data.

In [ ]:
#| export
@patch
def _update_indices(self:SemanticMemory, expanded_data):
    """Update basic 'by_id' and 'by_type' indices based on expanded JSON-LD data"""
    if not isinstance(expanded_data, list):
        # If input is not a list (e.g., single expanded node), wrap it in a list
        expanded_data = [expanded_data] if isinstance(expanded_data, dict) else []
        
    for node in expanded_data:
        if not isinstance(node, dict):
            continue
            
        # Index by ID
        node_id = node.get('@id')
        if node_id:
            # Store the node itself in the by_id index
            self.indices["by_id"][node_id] = node
            
        # Index by Type
        types = node.get('@type', [])
        if not isinstance(types, list): # Ensure types is always a list
            types = [types]
            
        for type_uri in types:
            if type_uri not in self.indices["by_type"]:
                self.indices["by_type"][type_uri] = []
            # Avoid adding duplicates if the same node is processed multiple times
            # Store the node itself, not just the ID
            if node not in self.indices["by_type"][type_uri]:
                 self.indices["by_type"][type_uri].append(node)

In [ ]:
#| export
@patch
def _update_indices_with_labels(self:SemanticMemory, expanded_data):
    """Update 'by_label' and 'by_description' indices"""
    # Common label properties (expanded form)
    label_properties = [
        RDFS.label.toPython(), # "http://www.w3.org/2000/01/rdf-schema#label"
        "http://www.w3.org/2004/02/skos/core#prefLabel",
        "http://schema.org/name",
        "http://purl.org/dc/terms/title"
    ]
    
    # Common description properties (expanded form)
    description_properties = [
        RDFS.comment.toPython(), # "http://www.w3.org/2000/01/rdf-schema#comment"
        "http://purl.org/dc/terms/description",
        "http://schema.org/description"
    ]
    
    # Ensure indices exist
    if "by_label" not in self.indices: self.indices["by_label"] = {}
    if "by_description" not in self.indices: self.indices["by_description"] = {}
    
    # Process expanded data
    if not isinstance(expanded_data, list):
        expanded_data = [expanded_data] if isinstance(expanded_data, dict) else []

    for node in expanded_data:
        if not isinstance(node, dict): continue
            
        node_id = node.get('@id')
        if not node_id: continue # Skip nodes without IDs for these indices
            
        # Extract and index labels
        for label_prop in label_properties:
            if label_prop in node:
                for value_obj in node[label_prop]:
                    if isinstance(value_obj, dict) and '@value' in value_obj:
                        label = value_obj['@value']
                        # Index by the exact label (lowercase for case-insensitivity?)
                        label_key = label.lower() # Or keep original case if needed
                        if label_key not in self.indices["by_label"]:
                            self.indices["by_label"][label_key] = []
                        # Store the node, avoid duplicates
                        if node not in self.indices["by_label"][label_key]:
                            self.indices["by_label"][label_key].append(node)
        
        # Extract and index descriptions (by keywords)
        for desc_prop in description_properties:
            if desc_prop in node:
                for value_obj in node[desc_prop]:
                    if isinstance(value_obj, dict) and '@value' in value_obj:
                        desc = value_obj['@value']
                        # Simple keyword extraction (split, lowercase, filter length)
                        keywords = [w.lower() for w in re.findall(r'\b\w{4,}\b', desc)] # Find words >= 4 chars
                        for keyword in set(keywords): # Use set to process each keyword once per node
                            if keyword not in self.indices["by_description"]:
                                self.indices["by_description"][keyword] = []
                            # Store the node, avoid duplicates
                            if node not in self.indices["by_description"][keyword]:
                                self.indices["by_description"][keyword].append(node)

# Querying Functions
Implement functions for retrieving data: `query_by_id`, `query_by_type`, `query_graph_entities`, `retrieve_in_context`, `search_with_contexts`. These allow querying by ID, type, graph membership, and text search while preserving context.

In [ ]:
#| export
@patch
def query_by_id(self:SemanticMemory, entity_id):
    """Retrieve an entity by its ID from the 'by_id' index"""
    # Check the main by_id index
    if entity_id in self.indices["by_id"]:
        return self.indices["by_id"][entity_id]
    
    # Check URI-to-URN mapping (if used)
    if hasattr(self, 'uri_to_urn') and entity_id in self.uri_to_urn:
        mapped_id = self.uri_to_urn[entity_id]
        if mapped_id in self.indices["by_id"]:
            return self.indices["by_id"][mapped_id]
            
    # Check URN-to-URI mapping (if used)
    if hasattr(self, 'urn_to_uri') and entity_id in self.urn_to_uri:
        mapped_id = self.urn_to_uri[entity_id]
        if mapped_id in self.indices["by_id"]:
            return self.indices["by_id"][mapped_id]
    
    # Entity not found in index
    return None

In [ ]:
#| export
@patch
def query_by_type(self:SemanticMemory, type_uri):
    """Retrieve all entities of a specific type from the 'by_type' index"""
    # Use the by_type index for efficiency
    if type_uri in self.indices["by_type"]:
        # Return the list of entities directly from the index
        return self.indices["by_type"][type_uri]
    else:
        # Type not found in index
        return []

    # # Fallback to RDF query (less efficient, kept for reference)
    # type_uri_ref = URIRef(type_uri)
    # entities = []
    # processed_subjects = set()
    #
    # # Find all subjects with the specified type in the default graph
    # for s in self.default_graph.subjects(RDF.type, type_uri_ref):
    #     subject_uri = str(s)
    #     if subject_uri in processed_subjects: continue
    #     processed_subjects.add(subject_uri)
    #
    #     # Try retrieving from by_id index first for complete data
    #     indexed_entity = self.query_by_id(subject_uri)
    #     if indexed_entity:
    #         entities.append(indexed_entity)
    #     else:
    #         # If not in index, reconstruct from RDF graph (less ideal)
    #         entity = {"@id": subject_uri, "@type": [type_uri]} # Assume list for type
    #         for p, o in self.default_graph.predicate_objects(s):
    #             pred = str(p)
    #             if pred == str(RDF.type): continue # Already handled
    #
    #             if pred not in entity: entity[pred] = []
    #
    #             value = None
    #             if isinstance(o, URIRef): value = {"@id": str(o)}
    #             elif isinstance(o, Literal):
    #                  value = {"@value": str(o)}
    #                  if o.language: value["@language"] = o.language
    #                  if o.datatype: value["@type"] = str(o.datatype)
    #
    #             if value and value not in entity[pred]: # Avoid duplicates
    #                  entity[pred].append(value)
    #         entities.append(entity)
    #
    # return entities

In [ ]:
#| export
@patch
def query_graph_entities(self:SemanticMemory, graph_id):
    """Get all entities that belong to a specific graph using graph tracking"""
    if graph_id not in self.graph_entities: return []
    
    entities = []
    entity_ids = self.graph_entities[graph_id]
    for entity_id in entity_ids:
        # Retrieve the full entity data using query_by_id
        entity_data = self.query_by_id(entity_id)
        if entity_data:
            entities.append(entity_data)
        # else: # Optionally log if an entity ID is tracked but not found
        #     print(f"Warning: Entity {entity_id} tracked in graph {graph_id} but not found in by_id index.")

    return entities

In [ ]:
#| export
@patch
def retrieve_in_context(self:SemanticMemory, entity_id, include_scoped_contexts=True):
    """
    Retrieve an entity with its original context structure preserved.
    
    Args:
        entity_id: The ID of the entity to retrieve.
        include_scoped_contexts: Whether to attempt to reconstruct scoped contexts (can be complex).
        
    Returns:
        The entity's original JSON-LD data (potentially modified if context was added), or None if not found.
    """
    # Check if we have the original data stored
    if entity_id not in self.original_data:
        # Maybe it was added directly to RDF without preserving original? Less likely with current flow.
        # As a fallback, try retrieving from by_id index, but context will be lost/expanded.
        expanded_entity = self.query_by_id(entity_id)
        if expanded_entity:
             print(f"Warning: Original data for {entity_id} not found. Returning expanded form.")
             # Attempt to compact it with a default context? Might be misleading.
             # For now, return the expanded form directly.
             return expanded_entity
        return None # Truly not found
    
    # Get the stored original data
    original_entity_data = copy.deepcopy(self.original_data[entity_id])

    # --- Handling Scoped Contexts (Simplified Approach) ---
    # The `_register_contexts` function already stores the structure.
    # The `add_jsonld` function stores the `original_data` which *should* include
    # the nested context definitions if they were present in the input.
    # Therefore, simply returning `original_entity_data` might be sufficient
    # if the input data correctly defined scoped contexts.

    # The complex reconstruction logic previously here might be unnecessary if
    # `original_data` correctly preserves the input structure including nested contexts.
    # Let's rely on `original_data` preservation.

    # If `include_scoped_contexts` is True (default), we assume the stored
    # `original_entity_data` contains the necessary structure.
    # If False, we might want to strip contexts, but that's usually not desired.
    # For now, `include_scoped_contexts` doesn't change the behavior,
    # relying on the stored original data.

    return original_entity_data

In [ ]:
#| export
@patch
def search_with_contexts(self:SemanticMemory, query, context_id=None, max_results=5):
    """
    Search for entities matching a query, returning results with original context.
    
    Args:
        query: Text to search for (case-insensitive).
        context_id: Optional context ID to limit search scope.
        max_results: Maximum number of results to return.
        
    Returns:
        List of matching entities, each retrieved using `retrieve_in_context`.
    """
    matching_entity_ids = set()
    query_lower = query.lower()

    # If a specific context is provided, search only entities using that context
    if context_id:
        if context_id in self.context_registry:
            entities_in_context = self.context_registry[context_id]["used_by"]
            for entity_id in entities_in_context:
                # Retrieve expanded entity for searching content
                entity_expanded = self.query_by_id(entity_id)
                if entity_expanded:
                    # Simple text search within the expanded entity's JSON string
                    entity_str = json.dumps(entity_expanded).lower()
                    if query_lower in entity_str:
                        matching_entity_ids.add(entity_id)
                        if len(matching_entity_ids) >= max_results: break
        else:
            print(f"Warning: Context ID {context_id} not found in registry.")
            return [] # Context not found, return empty results

    # If no context specified, search across all relevant indices
    else:
        # 1. Search by Label Index
        if "by_label" in self.indices:
            for label_key, entities in self.indices["by_label"].items():
                if query_lower in label_key: # Assumes label_key is already lowercase
                    for entity in entities:
                        if "@id" in entity:
                            matching_entity_ids.add(entity["@id"])
                            if len(matching_entity_ids) >= max_results: break
                    if len(matching_entity_ids) >= max_results: break
        
        # 2. Search by Description Keyword Index (if still need results)
        if len(matching_entity_ids) < max_results and "by_description" in self.indices:
             # Search for the query term as a keyword
             if query_lower in self.indices["by_description"]:
                 for entity in self.indices["by_description"][query_lower]:
                     if "@id" in entity:
                         matching_entity_ids.add(entity["@id"])
                         if len(matching_entity_ids) >= max_results: break

             # Also search keywords within the query against the index
             query_keywords = [w.lower() for w in re.findall(r'\b\w{4,}\b', query)]
             for keyword in query_keywords:
                 if len(matching_entity_ids) >= max_results: break
                 if keyword in self.indices["by_description"]:
                     for entity in self.indices["by_description"][keyword]:
                         if "@id" in entity:
                             matching_entity_ids.add(entity["@id"])
                             if len(matching_entity_ids) >= max_results: break


        # 3. Fallback: Full text search in expanded data (less efficient)
        if len(matching_entity_ids) < max_results:
            for entity_id, entity_expanded in self.indices["by_id"].items():
                if entity_id not in matching_entity_ids: # Avoid re-checking
                    entity_str = json.dumps(entity_expanded).lower()
                    if query_lower in entity_str:
                        matching_entity_ids.add(entity_id)
                        if len(matching_entity_ids) >= max_results: break

    # Retrieve each matching entity with its original context
    results = []
    for entity_id in list(matching_entity_ids)[:max_results]: # Ensure max_results limit
        entity_with_context = self.retrieve_in_context(entity_id)
        if entity_with_context:
            results.append(entity_with_context)
            
    return results

# Linked Data Operations
Implement functions for interacting with external linked data: `dereference_uri`, `follow_link`, `resolve_did`. These handle fetching data from URIs, following links, and resolving DIDs.

In [ ]:
#| export
@patch
def dereference_uri(self:SemanticMemory, uri, force_refresh=False, ttl_seconds=3600):
    """
    Dereference a URI, cache the result, and add it to memory.
    Handles content negotiation for JSON-LD and Turtle.
    
    Args:
        uri: The URI to dereference.
        force_refresh: Whether to force a refresh even if cached.
        ttl_seconds: Time-to-live in seconds for cache entries.
    
    Returns:
        The fetched JSON-LD data (as dict/list), or a dict with an error message.
    """
    now = datetime.now()
    
    # Check cache first
    if not force_refresh and uri in self.uri_cache:
        cache_entry = self.uri_cache[uri]
        # Check expiration
        if cache_entry.get("expires_at") and cache_entry["expires_at"] > now:
            # Return cached data, potentially re-adding to memory if needed?
            # For now, just return the data. Assumes it was added on first fetch.
            return cache_entry["data"]
        # Else: cache expired, proceed to fetch

    # Fetch the resource
    data = None
    format_detected = "unknown"
    status_code = 0
    error_message = None

    try:
        headers = {
            # Prioritize JSON-LD, then Turtle, then general JSON
            "Accept": "application/ld+json, text/turtle;q=0.9, application/json;q=0.8, */*;q=0.5"
        }
        
        response = self.client.get(uri, headers=headers, timeout=10.0) # Added timeout
        status_code = response.status_code
        response.raise_for_status() # Raise HTTPError for bad responses (4xx or 5xx)
            
        content_type = response.headers.get("Content-Type", "").split(";")[0].strip()
        
        # Process based on content type
        if "application/ld+json" in content_type:
            data = response.json()
            format_detected = "json-ld"
        elif "text/turtle" in content_type:
            g = Graph()
            g.parse(data=response.text, format="turtle", publicID=uri) # Provide publicID
            # Serialize to JSON-LD (might require context or be verbose)
            # Using context=None for basic conversion
            data = json.loads(g.serialize(format="json-ld", context=None, indent=2))
            format_detected = "turtle"
        elif "application/json" in content_type: # Handle plain JSON as potential JSON-LD
             try:
                 potential_jsonld = response.json()
                 # Basic check if it looks like JSON-LD (has @context or @id/@graph)
                 if isinstance(potential_jsonld, dict) and \
                    ('@context' in potential_jsonld or '@id' in potential_jsonld or '@graph' in potential_jsonld):
                      data = potential_jsonld
                      format_detected = "json-ld" # Assume it's JSON-LD
                 else:
                      # Treat as non-linked data, maybe wrap it?
                      print(f"Warning: Received plain JSON for {uri}. Treating as non-linked data.")
                      data = {"@id": uri, "value": potential_jsonld} # Simple wrapping
                      format_detected = "json"
             except json.JSONDecodeError:
                 error_message = "Failed to decode JSON response"
                 format_detected = "error"
        else:
            # Unsupported content type
            error_message = f"Unsupported Content-Type: {content_type}"
            format_detected = "unsupported"
            data = {"@id": uri, "error": error_message, "contentType": content_type}

    except httpx.RequestError as e:
        error_message = f"HTTP Request failed: {e}"
        format_detected = "error"
    except json.JSONDecodeError as e:
        error_message = f"JSON Decode failed: {e}"
        format_detected = "error"
    except Exception as e: # Catch other potential errors (RDF parsing, etc.)
        error_message = f"Dereferencing failed: {e}"
        format_detected = "error"

    # Prepare cache entry data
    if error_message:
        cache_data = {"@id": uri, "error": error_message}
        expires_at = now + timedelta(seconds=min(ttl_seconds, 300)) # Shorter TTL for errors
    else:
        cache_data = data
        expires_at = now + timedelta(seconds=ttl_seconds)

    # Update cache
    self.uri_cache[uri] = {
        "data": cache_data,
        "format": format_detected,
        "fetched_at": now,
        "expires_at": expires_at,
        "status": status_code
    }
    
    # If data was successfully fetched and processed into JSON-LD format, add it to memory
    if data and not error_message and format_detected in ["json-ld", "turtle", "json"]:
        # Use add_jsonld_with_graph to handle potential @graph structures
        self.add_jsonld_with_graph(data) # This handles context registration, indexing, RDF addition

    return cache_data # Return the data (or error dict)

In [ ]:
#| export
@patch
def follow_link(self:SemanticMemory, uri, predicate=None, limit=5):
    """
    Follow links (object properties) from a resource URI.
    Dereferences the starting URI if not already in memory, then finds linked URIs.
    Optionally dereferences the linked URIs.
    
    Args:
        uri: Starting URI.
        predicate: Optional specific predicate URI (expanded form) to follow.
        limit: Maximum number of unique linked URIs to return.
    
    Returns:
        List of unique URIs linked from the starting URI via the specified predicate (or any predicate).
    """
    # 1. Ensure we have data about the starting URI in expanded form
    start_entity_expanded = self.query_by_id(uri)
    if not start_entity_expanded:
        # If not in index, try dereferencing it
        print(f"Dereferencing starting URI: {uri}")
        deref_result = self.dereference_uri(uri)
        if "error" in deref_result:
            print(f"Failed to dereference starting URI {uri}: {deref_result['error']}")
            return []
        # After dereferencing, it should be in the index
        start_entity_expanded = self.query_by_id(uri)
        if not start_entity_expanded:
             print(f"Error: Could not find expanded data for {uri} even after dereferencing.")
             return []

    linked_uris = set()
    
    # The entity data should be a dictionary after expansion
    if not isinstance(start_entity_expanded, dict):
         print(f"Warning: Expanded data for {uri} is not a dictionary.")
         return []

    # Iterate through properties of the expanded entity
    for pred, values in start_entity_expanded.items():
        # Skip non-property keys like @id, @type, @index etc.
        if pred.startswith('@'): continue
            
        # Filter by predicate if specified
        if predicate and pred != predicate: continue
            
        # Values should be a list after expansion
        if not isinstance(values, list): continue

        # Look for linked objects (dictionaries with @id)
        for value_obj in values:
            if isinstance(value_obj, dict) and '@id' in value_obj:
                target_uri = value_obj['@id']
                # Add to set (automatically handles uniqueness)
                linked_uris.add(target_uri)
                if len(linked_uris) >= limit: break # Stop if limit reached
        if len(linked_uris) >= limit: break # Stop if limit reached

    # Optional: Dereference the found URIs (could be a separate step/option)
    # for target_uri in linked_uris:
    #     self.dereference_uri(target_uri) # Dereference each linked URI

    return list(linked_uris)

In [ ]:
#| export
@patch
def resolve_did(self:SemanticMemory, did):
    """Resolve a DID to its DID Document (Placeholder/Simulation)"""
    # Check if we already have this DID Document in our memory
    did_doc = self.query_by_id(did)
    if did_doc:
        # Basic check if it looks like a DID doc
        if isinstance(did_doc, dict) and did_doc.get('@id') == did and '@context' in did_doc:
             return did_doc
        # Else: Found something with the ID, but might not be the DID doc itself. Overwrite?

    print(f"Simulating DID resolution for: {did}")
    # In a real implementation: use a DID resolver library (e.g., pydid)
    # result = did_resolver.resolve(did)
    # For now, simulate a basic DID document structure

    did_document = {
        "@context": [
            "https://www.w3.org/ns/did/v1",
            "https://w3id.org/security/suites/ed25519-2020/v1" # Example context for key type
        ],
        "id": did,
        "verificationMethod": [
            {
                "id": f"{did}#keys-1",
                "type": "Ed25519VerificationKey2020", # Updated type example
                "controller": did,
                "publicKeyMultibase": "z6Mkg..." # Placeholder for actual key data
            }
        ],
        "authentication": [f"{did}#keys-1"],
        "assertionMethod": [f"{did}#keys-1"]
        # Add other DID doc components like service endpoints if needed
    }
    
    # Add the simulated DID document to our memory
    self.add_jsonld(did_document)
    
    # Return the simulated document
    return did_document

# JSON-LD Container Building Functions
Implement functions to create JSON-LD 1.1 containers: `build_type_container`, `build_property_container`. These generate structures for efficient type-based or property-based access.

In [ ]:
#| export
@patch
def build_type_container(self:SemanticMemory, base_type=None, include_subtypes=False):
    """
    Build a JSON-LD 1.1 @type container for entities in memory.
    
    Args:
        base_type: Optional base type URI to filter by.
        include_subtypes: Whether to include subtypes of the base_type (requires RDFS reasoning or pre-computation).
                          Currently NOT implemented robustly.
    
    Returns:
        A JSON-LD document structured with `@container: @type`.
    """
    # Define the container structure
    container_id = "http://memory.org/containers/byType"
    container = {
        "@context": {
            "@version": 1.1,
            # Define a term for the resources, mapping it to the container ID
            # and specifying the container type and index property (@type)
            "resourcesByType": {
                "@id": container_id,
                "@container": "@type"
            }
            # We might need to include vocabularies used in type URIs if not absolute
        },
        # Use the term defined in the context as the key
        "resourcesByType": {}
    }
    
    target_types = set()
    if base_type:
        target_types.add(base_type)
        if include_subtypes:
            # --- Subtype Expansion (Basic RDF Query - potentially slow) ---
            # This requires RDFS reasoning capabilities or pre-computed hierarchy.
            # Simple query against the default graph:
            print("Warning: Subtype inclusion is experimental and may be slow.")
            subclass_prop = RDFS.subClassOf
            q = """
                PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
                SELECT ?sub WHERE {
                    ?sub rdfs:subClassOf* <%s> .
                }
            """ % base_type
            try:
                 results = self.default_graph.query(q)
                 for row in results:
                     target_types.add(str(row.sub))
            except Exception as e:
                 print(f"Error querying subtypes for {base_type}: {e}")
            # --- End Subtype Expansion ---

    # Iterate through the 'by_type' index
    for type_uri, entities in self.indices["by_type"].items():
        # Filter by target types if specified
        if base_type and type_uri not in target_types:
            continue
            
        # Add entities to the container under their type URI
        if type_uri not in container["resourcesByType"]:
            container["resourcesByType"][type_uri] = []
            
        for entity_node in entities:
            entity_id = entity_node.get('@id')
            if entity_id:
                # Add a reference to the entity
                # Avoid adding duplicates if an entity has multiple relevant types
                if {"@id": entity_id} not in container["resourcesByType"][type_uri]:
                     container["resourcesByType"][type_uri].append({"@id": entity_id})

    # Optional: Add the container itself to memory?
    # self.add_jsonld(container) # This might create recursive structures if not careful

    return container

In [ ]:
#| export
@patch
def build_property_container(self:SemanticMemory, property_uri, index_property_uri=None):
    """
    Build a JSON-LD 1.1 @index container based on the values of a property.
    
    Args:
        property_uri: The property whose values will be used as keys in the index. (Expanded form)
        index_property_uri: Optional. If provided, the container indexes entities *by* this property's value.
                           If None, it indexes the values of `property_uri` themselves.
                           This needs clarification based on desired output. Let's assume we index *entities*
                           by the value of `property_uri`.
    
    Returns:
        A JSON-LD document structured with `@container: @index`.
    """
    if index_property_uri is None: index_property_uri = property_uri # Default to indexing by the property itself

    container_id = f"http://memory.org/containers/byIndex/{index_property_uri.split('/')[-1]}"
    container = {
        "@context": {
            "@version": 1.1,
            # Define a term for the resources
            "resourcesByIndex": {
                "@id": container_id,
                "@container": "@index",
                "@index": index_property_uri # Specify the property to index on
            }
            # Include relevant vocabularies if URIs are not absolute
        },
        "resourcesByIndex": {}
    }
    
    # Iterate through all entities in the 'by_id' index
    for entity_id, entity_node in self.indices["by_id"].items():
        if not isinstance(entity_node, dict): continue

        # Check if the entity has the property_uri
        if property_uri in entity_node:
            values = entity_node[property_uri]
            if not isinstance(values, list): values = [values] # Ensure list

            for value_obj in values:
                index_key = None
                # Extract the value to be used as the index key
                if isinstance(value_obj, dict):
                    if '@value' in value_obj:
                        index_key = str(value_obj['@value']) # Use literal value
                    elif '@id' in value_obj:
                         index_key = value_obj['@id'] # Use linked entity ID
                # Handle simple string values if expansion didn't create objects (less likely)
                elif isinstance(value_obj, str):
                     index_key = value_obj

                if index_key is not None:
                    # Add the entity to the container under this index key
                    if index_key not in container["resourcesByIndex"]:
                        container["resourcesByIndex"][index_key] = []
                    
                    # Add a reference to the entity, avoid duplicates
                    entity_ref = {"@id": entity_id}
                    if entity_ref not in container["resourcesByIndex"][index_key]:
                         container["resourcesByIndex"][index_key].append(entity_ref)

    return container

# Agent Integration Functions
Implement functions to integrate with an LLM agent (like Claude): `retrieve_relevant_memory`, `query_translator`, `llm_query_translator`, `llm_process_information`, `answer_query_with_context_aware_tools`, `answer_query_with_graph_aware_tools`. These handle query translation, memory retrieval, information processing, and tool usage.

In [ ]:
#| export
@patch
def retrieve_relevant_memory(self:SemanticMemory, structured_query, max_results=5):
    """
    Retrieve relevant information from memory based on a structured query.
    Prioritizes ID, Type, Label, Keyword matches. Returns entities with context. (Corrected Label Matching)
    
    Args:
        structured_query: Dictionary from llm_query_translator 
                          (e.g., {"entities": [], "types": [], "properties": [], "keywords": []})
        max_results: Maximum number of results to return.
        
    Returns:
        List of relevant entities, retrieved using `retrieve_in_context`.
    """
    relevant_entity_ids = set()
    
    # Extract components from the structured query
    entity_ids_query = structured_query.get("entities", [])
    types_query = structured_query.get("types", []) 
    properties_query = structured_query.get("properties", []) # Currently unused in retrieval logic, but could be
    keywords_query = structured_query.get("keywords", [])
    
    # Clean keywords
    clean_keywords = {re.sub(r'[^\w\s-]', '', keyword.lower()) for keyword in keywords_query if keyword}
    
    # Helper function to retrieve and format results
    def get_results():
        results = []
        for entity_id in list(relevant_entity_ids)[:max_results]:
            entity_with_context = self.retrieve_in_context(entity_id)
            if entity_with_context: results.append(entity_with_context)
        return results

    # 1. Direct Entity ID Matches
    for entity_id in entity_ids_query:
        if self.query_by_id(entity_id): # Check if ID exists in index
            relevant_entity_ids.add(entity_id)
            if len(relevant_entity_ids) >= max_results: return get_results()

    # 2. Type Matches
    for type_uri in types_query:
        entities_of_type = self.query_by_type(type_uri) # Returns list of expanded entities
        for entity in entities_of_type:
            if entity and '@id' in entity:
                 entity_id = entity['@id']
                 if entity_id not in relevant_entity_ids:
                     relevant_entity_ids.add(entity_id)
                     if len(relevant_entity_ids) >= max_results: return get_results()
        # No need for outer break here, inner break handles max_results

    # 3. Label Matches (Corrected Logic: check if keyword is IN label_key)
    if "by_label" in self.indices:
        label_keys = list(self.indices["by_label"].keys()) # Iterate safely
        for label_key in label_keys:
            entities = self.indices["by_label"][label_key]
            for keyword in clean_keywords:
                if keyword in label_key: # Check if keyword is part of the indexed label
                    for entity in entities:
                        if entity and '@id' in entity:
                             entity_id = entity['@id']
                             if entity_id not in relevant_entity_ids:
                                 relevant_entity_ids.add(entity_id)
                                 if len(relevant_entity_ids) >= max_results: return get_results()
                    # No need for break here, check all keywords against this label_key
            # No need for break here, check all label_keys
    
    # 4. Description Keyword Matches (using keywords against by_description index)
    if "by_description" in self.indices:
        desc_keys = list(self.indices["by_description"].keys()) # Iterate safely
        for keyword in clean_keywords: # Iterate through search keywords
             if keyword in self.indices["by_description"]: # Check if keyword exists as an index key
                 for entity in self.indices["by_description"][keyword]:
                     if entity and '@id' in entity:
                          entity_id = entity['@id']
                          if entity_id not in relevant_entity_ids:
                              relevant_entity_ids.add(entity_id)
                              if len(relevant_entity_ids) >= max_results: return get_results()
             # No need for outer break, check all keywords

    # 5. Fallback: Keyword match against entity IDs themselves
    all_entity_ids = list(self.indices["by_id"].keys()) # Iterate safely
    for entity_id in all_entity_ids:
         if entity_id not in relevant_entity_ids: # Only check if not already found
             eid_lower = entity_id.lower()
             for keyword in clean_keywords:
                 if keyword in eid_lower:
                     relevant_entity_ids.add(entity_id)
                     if len(relevant_entity_ids) >= max_results: return get_results()
                     break # Found a match for this entity_id, move to next id

    # Final retrieval with context
    return get_results()

In [ ]:
#| export
@patch
def query_translator(self:SemanticMemory, natural_language_query):
    """
    Translate a natural language query into structured memory queries (Simple Rule-Based Version).
    Prefer `llm_query_translator` for better results.
    
    Args:
        natural_language_query: A natural language question or request.
        
    Returns:
        Dictionary with keys: "entities", "types", "properties", "keywords".
    """
    structured_query = {
        "entities": [],
        "types": [],
        "properties": [], # Simple version doesn't extract properties well
        "keywords": []
    }
    
    # Simple keyword extraction
    words = re.findall(r'\b\w+\b', natural_language_query.lower())
    stopwords = {'what', 'is', 'who', 'where', 'when', 'how', 'the', 'a', 'an', 'of', 'in', 'at', 'for', 'to', 'and', 'or'}
    keywords = [w for w in words if w not in stopwords and len(w) > 2]
    structured_query["keywords"] = list(set(keywords)) # Unique keywords

    # Simple entity/type matching (very basic)
    # Check if keywords match known entity IDs (or parts of them)
    for keyword in keywords:
        # Check labels first
        if keyword in self.indices.get("by_label", {}):
             for entity in self.indices["by_label"][keyword]:
                 if entity and "@id" in entity:
                     structured_query["entities"].append(entity["@id"])
        # Check IDs containing the keyword
        for entity_id in self.indices["by_id"].keys():
            if keyword in entity_id.lower():
                structured_query["entities"].append(entity_id)
                
    # Check if keywords match known types (or parts of them)
    for keyword in keywords:
        for type_uri in self.indices["by_type"].keys():
            # Match end part of URI or full keyword match
            type_name = type_uri.split('/')[-1].split('#')[-1].lower()
            if keyword == type_name or keyword in type_uri.lower():
                structured_query["types"].append(type_uri)

    # Remove duplicates
    structured_query["entities"] = list(set(structured_query["entities"]))
    structured_query["types"] = list(set(structured_query["types"]))

    return structured_query

In [ ]:
#| eval: false
# Re-run tests with order-independent checks

# Setup memory and data again for isolated testing
memory_test = SemanticMemory()
person_data = {
    "@context": {"@vocab": "http://schema.org/"},
    "@id": "http://example.org/person/alice",
    "@type": "Person",
    "name": "Alice Smith",
    "description": "A software engineer living in Wonderland."
}
org_data = {
    "@context": {"@vocab": "http://schema.org/"},
    "@id": "http://example.org/org/wonderland-inc",
    "@type": "Organization",
    "name": "Wonderland Inc.",
    "description": "A fictional company specializing in software."
}
memory_test.add_jsonld(person_data)
memory_test.add_jsonld(org_data)

# Define the manual structured query again
structured_query_manual = {
    "entities": ["http://example.org/person/alice"],
    "types": ["http://schema.org/Person"],
    "properties": [],
    "keywords": ["alice", "software", "engineer"]
}

# Retrieve based on the manual query
retrieved_manual = memory_test.retrieve_relevant_memory(structured_query_manual, max_results=3)
print("\nRetrieved based on Manual Query (Re-test):")
for item in retrieved_manual: print(json.dumps(item, indent=2))

# --- Revised Tests for Manual Query Retrieval ---
# Check the number of results. Keywords "software" and "engineer" match both entities.
# Even though Alice is specified by ID and Type, the keyword matching phase adds the Org.
test_eq(len(retrieved_manual), 2) # Expect exactly two results (Alice and Org)

# Check that both expected IDs are present in the results (order doesn't matter)
retrieved_manual_ids = {item['@id'] for item in retrieved_manual}
test_eq(retrieved_manual_ids, {"http://example.org/person/alice", "http://example.org/org/wonderland-inc"})

# Find Alice's data specifically within the results and check a property
alice_retrieved = next((item for item in retrieved_manual if item['@id'] == "http://example.org/person/alice"), None)
test_ne(alice_retrieved, None) # Ensure Alice was found
test_eq(alice_retrieved['name'], "Alice Smith") # Check a property of Alice

# --- Re-test Retrieval based on 'software' keyword ---
structured_query_software = {"entities": [], "types": [], "properties": [], "keywords": ["software"]}
retrieved_software = memory_test.retrieve_relevant_memory(structured_query_software, max_results=3)
print("\nRetrieved based on 'software' keyword (Re-test):")
for item in retrieved_software: print(json.dumps(item, indent=2))

# Check if both Alice and Wonderland Inc. are found (order might vary)
retrieved_ids_software = {item['@id'] for item in retrieved_software}
test_eq(retrieved_ids_software, {"http://example.org/person/alice", "http://example.org/org/wonderland-inc"})
test_eq(len(retrieved_software), 2) # Ensure exactly two results were found


Retrieved based on Manual Query (Re-test):
{
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@id": "http://example.org/person/alice",
  "@type": "Person",
  "name": "Alice Smith",
  "description": "A software engineer living in Wonderland."
}
{
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@id": "http://example.org/org/wonderland-inc",
  "@type": "Organization",
  "name": "Wonderland Inc.",
  "description": "A fictional company specializing in software."
}

Retrieved based on 'software' keyword (Re-test):
{
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@id": "http://example.org/person/alice",
  "@type": "Person",
  "name": "Alice Smith",
  "description": "A software engineer living in Wonderland."
}
{
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@id": "http://example.org/org/wonderland-inc",
  "@type": "Organization",
  "name": "Wonderland Inc.",
  "description": "A fictional company specializing in software."
}


In [ ]:
import inspect

#| export
import fastcore.test

def show_fastcore_test_methods():
    "List available functions in the fastcore.test module"
    methods = [name for name, obj in inspect.getmembers(fastcore.test) if callable(obj)]
    return methods

# Example usage:
available_tests = show_fastcore_test_methods()
print("Available fastcore.test methods:")
print(available_tests)

Available fastcore.test methods:
['BuiltinFunctionType', 'BuiltinMethodType', 'Counter', 'Dict', 'ExceptionExpected', 'FunctionType', 'Generator', 'Iterable', 'Iterator', 'List', 'MethodDescriptorType', 'MethodType', 'MethodWrapperType', 'NoneType', 'Optional', 'Path', 'Sequence', 'Set', 'SimpleNamespace', 'Tuple', 'Union', 'WrapperDescriptorType', 'all_equal', 'any_is_instance', 'array_equal', 'attrgetter', 'df_equal', 'equals', 'in_colab', 'in_ipython', 'in_jupyter', 'in_notebook', 'ipython_shell', 'is_close', 'is_coll', 'is_iter', 'isinstance_str', 'itemgetter', 'nequals', 'noop', 'noops', 'partial', 'redirect_stdout', 'reduce', 'remove_prefix', 'remove_suffix', 'test', 'test_close', 'test_eq', 'test_eq_type', 'test_fail', 'test_fig_exists', 'test_is', 'test_ne', 'test_shuffled', 'test_stdout', 'test_warns', 'warn']


In [ ]:
#| eval: false

# Add more data for specific tests
project_data = {
    "@context": {"@vocab": "http://schema.org/"},
    "@id": "http://example.org/project/secret",
    "@type": "Project",
    "name": "Project Chimera", # Keyword 'chimera' only in name
    "description": "A top-secret initiative."
}
memory_test.add_jsonld(project_data)

# --- Test: Keyword Matches Label Only ---
structured_query_label = {"entities": [], "types": [], "properties": [], "keywords": ["chimera"]}
retrieved_label = memory_test.retrieve_relevant_memory(structured_query_label, max_results=3)
print("\nRetrieved based on 'chimera' keyword (Label Match):")
for item in retrieved_label: print(json.dumps(item, indent=2))

# Check if only the project is found
retrieved_ids_label = {item['@id'] for item in retrieved_label}
test_eq(retrieved_ids_label, {"http://example.org/project/secret"})
test_eq(len(retrieved_label), 1)

# --- Test: No Matches Found ---
structured_query_nomatch = {"entities": [], "types": [], "properties": [], "keywords": ["nonexistent", "xyzzy"]}
retrieved_nomatch = memory_test.retrieve_relevant_memory(structured_query_nomatch, max_results=3)
print("\nRetrieved based on non-matching keywords:")
print(retrieved_nomatch)

# Check if the result is an empty list
test_eq(retrieved_nomatch, [])

# --- Test: max_results Enforcement (Example Setup) ---
# Add a few more entities matching 'software'
for i in range(5):
    memory_test.add_jsonld({
        "@context": {"@vocab": "http://schema.org/"},
        "@id": f"http://example.org/product/{i}",
        "@type": "SoftwareApplication",
        "name": f"Software {i}",
        "description": "Generic software product."
    })

structured_query_software_limit = {"entities": [], "types": [], "properties": [], "keywords": ["software"]}
# Now Alice, Org, and 5 products match 'software' (7 total)
retrieved_software_limit = memory_test.retrieve_relevant_memory(structured_query_software_limit, max_results=3)
print(f"\nRetrieved based on 'software' keyword (max_results=3): {len(retrieved_software_limit)} items")
# Check that the limit was enforced
test_eq(len(retrieved_software_limit), 3)
# Verify the original two are likely included (order isn't guaranteed, but they match description/label)
retrieved_ids_limit = {item['@id'] for item in retrieved_software_limit}
# This check might be fragile depending on retrieval order, but demonstrates the idea
# assert "http://example.org/person/alice" in retrieved_ids_limit
# assert "http://example.org/org/wonderland-inc" in retrieved_ids_limit



Retrieved based on 'chimera' keyword (Label Match):
{
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@id": "http://example.org/project/secret",
  "@type": "Project",
  "name": "Project Chimera",
  "description": "A top-secret initiative."
}

Retrieved based on non-matching keywords:
[]

Retrieved based on 'software' keyword (max_results=3): 3 items


In [ ]:
#| eval: false

# --- Test: Retrieval by Specific Type Only ---
structured_query_org_type = {
    "entities": [],
    "types": ["http://schema.org/Organization"],
    "properties": [],
    "keywords": []
}
retrieved_org_type = memory_test.retrieve_relevant_memory(structured_query_org_type, max_results=3)
print("\nRetrieved based on Organization type:")
for item in retrieved_org_type: print(json.dumps(item, indent=2))

# Check if only Wonderland Inc. is found
retrieved_ids_org_type = {item['@id'] for item in retrieved_org_type}
test_eq(retrieved_ids_org_type, {"http://example.org/org/wonderland-inc"})
test_eq(len(retrieved_org_type), 1)

# --- Test: Retrieval by Specific ID Only ---
structured_query_alice_id = {
    "entities": ["http://example.org/person/alice"],
    "types": [],
    "properties": [],
    "keywords": []
}
retrieved_alice_id = memory_test.retrieve_relevant_memory(structured_query_alice_id, max_results=3)
print("\nRetrieved based on Alice's ID:")
for item in retrieved_alice_id: print(json.dumps(item, indent=2))

# Check if only Alice is found
retrieved_ids_alice = {item['@id'] for item in retrieved_alice_id}
test_eq(retrieved_ids_alice, {"http://example.org/person/alice"})
test_eq(len(retrieved_alice_id), 1)

# --- Test: Combining Type and Keyword ---
# Query for Organizations with 'fictional' keyword
structured_query_org_keyword = {
    "entities": [],
    "types": ["http://schema.org/Organization"],
    "properties": [],
    "keywords": ["fictional"]
}
retrieved_org_keyword = memory_test.retrieve_relevant_memory(structured_query_org_keyword, max_results=3)
print("\nRetrieved based on Organization type and 'fictional' keyword:")
for item in retrieved_org_keyword: print(json.dumps(item, indent=2))

# Check if only Wonderland Inc. is found (matches both type and keyword in description)
retrieved_ids_org_keyword = {item['@id'] for item in retrieved_org_keyword}
test_eq(retrieved_ids_org_keyword, {"http://example.org/org/wonderland-inc"})
test_eq(len(retrieved_org_keyword), 1)


# --- Test: Keyword Case Sensitivity (Should be Case-Insensitive) ---
# Query using uppercase keyword "SOFTWARE"
structured_query_software_upper = {
    "entities": [],
    "types": [],
    "properties": [],
    "keywords": ["SOFTWARE"]
}
retrieved_software_upper = memory_test.retrieve_relevant_memory(structured_query_software_upper, max_results=3)
print("\nRetrieved based on 'SOFTWARE' keyword (Uppercase):")
# Check the actual results based on label match priority
retrieved_ids_software_upper = {item['@id'] for item in retrieved_software_upper}
# Expected: The first 3 entities matching "software" in the label index
expected_ids_upper = {f"http://example.org/product/{i}" for i in range(3)} # Assuming products 0, 1, 2 are found first
test_eq(retrieved_ids_software_upper, expected_ids_upper)
test_eq(len(retrieved_software_upper), 3) # Function correctly returns max_results



# --- Test: Keyword in ID (Fallback Match) ---
# Add an entity where the keyword only appears in the ID
memory_test.add_jsonld({
    "@context": {"@vocab": "http://schema.org/"},
    "@id": "http://example.org/special-software-tool",
    "@type": "WebApplication",
    "name": "Web Tool",
    "description": "A useful online utility."
})
structured_query_id_keyword = {
    "entities": [],
    "types": [],
    "properties": [],
    "keywords": ["special"]
}
retrieved_id_keyword = memory_test.retrieve_relevant_memory(structured_query_id_keyword, max_results=3)
print("\nRetrieved based on 'special' keyword (ID match):")
for item in retrieved_id_keyword: print(json.dumps(item, indent=2))
# Check if the new tool is found
retrieved_ids_id_keyword = {item['@id'] for item in retrieved_id_keyword}
test_eq(retrieved_ids_id_keyword, {"http://example.org/special-software-tool"})
test_eq(len(retrieved_id_keyword), 1)



Retrieved based on Organization type:
{
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@id": "http://example.org/org/wonderland-inc",
  "@type": "Organization",
  "name": "Wonderland Inc.",
  "description": "A fictional company specializing in software."
}

Retrieved based on Alice's ID:
{
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@id": "http://example.org/person/alice",
  "@type": "Person",
  "name": "Alice Smith",
  "description": "A software engineer living in Wonderland."
}

Retrieved based on Organization type and 'fictional' keyword:
{
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@id": "http://example.org/org/wonderland-inc",
  "@type": "Organization",
  "name": "Wonderland Inc.",
  "description": "A fictional company specializing in software."
}

Retrieved based on 'SOFTWARE' keyword (Uppercase):

Retrieved based on 'special' keyword (ID match):
{
  "@context": {
    "@vocab": "http://schema.org/"
  },
  "@id": "http://example.org/spec